# Silver Layer — ICU Stays

## Source
`cdr_mimiciv.bronze.icustays`

## Target
`cdr_mimiciv.silver.icustays`

The target table is published with CREATE OR REPLACE semantics; downstream Gold depends on it.

## Grain
One row per ICU stay, identified by `stay_id` (official MIMIC-IV icu module).

## Reference tables read
`<target_schema>.patients` and `<target_schema>.admissions` (referential validation, and
enforced as publication gates under `enforce_profiled_counts`). Run those notebooks first.

## Principles
Typed, row-preserving conformance of source fields; no flags, no corrections, no derived
analytical fields. Profiled expectation: 94,458 rows; no cast failures, reversed
timestamps, negative `los` or `los` / timestamp mismatches over one minute.

In [0]:
from pyspark.sql import functions as F
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

SOURCE = f"{BRONZE}.icustays"
TARGET = f"{SILVER}.icustays"

In [0]:
# ============================================================
# STEP 1: BRONZE SOURCE INSPECTION
# ============================================================

# Load existing Bronze data
icu_bronze = spark.table(SOURCE)

# Inspect source structure
print("SOURCE:", SOURCE)
icu_bronze.printSchema()

# Check row count and ICU stay identifiers
profile = icu_bronze.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("stay_id").alias("unique_stay_ids"),
    F.sum(
        F.when(F.col("stay_id").isNull(), 1).otherwise(0)
    ).alias("null_stay_ids")
)

display(profile)

# Preview actual ICU stay records
display(icu_bronze.limit(5))


SOURCE: cdr_mimiciv.bronze.icustays
root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- stay_id: integer (nullable = true)
 |-- first_careunit: string (nullable = true)
 |-- last_careunit: string (nullable = true)
 |-- intime: timestamp (nullable = true)
 |-- outtime: timestamp (nullable = true)
 |-- los: double (nullable = true)



total_rows,unique_stay_ids,null_stay_ids
94458,94458,0


subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23T14:00:00.000Z,2180-07-23T23:50:47.000Z,0.4102662037037037
10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02T19:37:00.000Z,2150-11-06T17:03:17.000Z,3.8932523148148146
10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27T08:42:00.000Z,2189-06-27T20:38:27.000Z,0.4975347222222222
10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20T19:18:02.000Z,2157-11-21T22:08:00.000Z,1.1180324074074075
10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19T15:42:24.000Z,2157-12-20T14:27:41.000Z,0.948113425925926


In [0]:

# ============================================================
# STEP 2: ICU DATA TYPE AND QUALITY VALIDATION
# Read-only: no tables are modified
# ============================================================

quality_check = spark.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT_IF(
        subject_id IS NOT NULL
        AND TRY_CAST(TRIM(subject_id) AS BIGINT) IS NULL
    ) AS invalid_subject_ids,

    COUNT_IF(
        hadm_id IS NOT NULL
        AND TRY_CAST(TRIM(hadm_id) AS BIGINT) IS NULL
    ) AS invalid_hadm_ids,

    COUNT_IF(
        stay_id IS NOT NULL
        AND TRY_CAST(TRIM(stay_id) AS BIGINT) IS NULL
    ) AS invalid_stay_ids,

    COUNT_IF(
        intime IS NOT NULL
        AND TRY_CAST(TRIM(intime) AS TIMESTAMP) IS NULL
    ) AS invalid_intime,

    COUNT_IF(
        outtime IS NOT NULL
        AND TRY_CAST(TRIM(outtime) AS TIMESTAMP) IS NULL
    ) AS invalid_outtime,

    COUNT_IF(
        los IS NOT NULL
        AND TRY_CAST(TRIM(los) AS DOUBLE) IS NULL
    ) AS invalid_los,

    COUNT_IF(
        TRY_CAST(intime AS TIMESTAMP) >
        TRY_CAST(outtime AS TIMESTAMP)
    ) AS reversed_timestamps,

    COUNT_IF(
        TRY_CAST(los AS DOUBLE) < 0
    ) AS negative_los

FROM {SOURCE}
""")

display(quality_check)


total_rows,invalid_subject_ids,invalid_hadm_ids,invalid_stay_ids,invalid_intime,invalid_outtime,invalid_los,reversed_timestamps,negative_los
94458,0,0,0,0,0,0,0,0


In [0]:

# ============================================================
# ICU SILVER — ICUSTAYS
# STEP 3: TYPED TRANSFORMATION + VALIDATION CONTRACT
# ============================================================

from pyspark.sql import functions as F

# Official MIMIC source-aligned datatypes
cast_types = {
    "subject_id": "INT",
    "hadm_id": "INT",
    "stay_id": "INT",
    "intime": "TIMESTAMP",
    "outtime": "TIMESTAMP",
    "los": "DOUBLE"
}

# Retain original values temporarily for audit comparisons
prepared = icu_bronze.select(
    *[
        F.col(c).alias(f"_raw_{c}")
        for c in cast_types
    ],
    *[
        F.expr(
            f"TRY_CAST(TRIM(`{c}`) AS {dtype})"
        ).alias(c)
        for c, dtype in cast_types.items()
    ],
    F.col("first_careunit"),
    F.col("last_careunit")
)

# One grouped validation pass
checks = [
    F.count("*").alias("total_rows"),
    F.countDistinct("stay_id").alias("unique_stays"),
    F.count_if(F.col("stay_id").isNull())
        .alias("null_stays")
]

# Detect any non-NULL source value lost during conversion
for c in cast_types:
    checks.append(
        F.count_if(
            F.col(f"_raw_{c}").isNotNull()
            & F.col(c).isNull()
        ).alias(f"cast_fail_{c}")
    )

# Check timeline consistency
checks.extend([
    F.count_if(
        F.col("intime") > F.col("outtime")
    ).alias("reversed_times"),

    F.count_if(
        F.col("los") < 0
    ).alias("negative_los"),

    # Compare recorded LOS against actual timestamp duration
    # Flag differences exceeding 1 minute
    F.count_if(
        F.col("los").isNotNull()
        & F.col("intime").isNotNull()
        & F.col("outtime").isNotNull()
        & (
            F.abs(
                F.col("los")
                - F.expr(
                    "timestampdiff(SECOND, intime, outtime)"
                ) / F.lit(86400.0)
            ) > F.lit(1.0 / 1440.0)
        )
    ).alias("los_mismatch_over_1min")
])

validation = prepared.agg(*checks)

display(validation)

# Final Silver column projection — no write yet
icu_silver = prepared.select(
    "subject_id",
    "hadm_id",
    "stay_id",
    "first_careunit",
    "last_careunit",
    "intime",
    "outtime",
    "los"
)

print("Silver candidate prepared — NOT published")
icu_silver.printSchema()


total_rows,unique_stays,null_stays,cast_fail_subject_id,cast_fail_hadm_id,cast_fail_stay_id,cast_fail_intime,cast_fail_outtime,cast_fail_los,reversed_times,negative_los,los_mismatch_over_1min
94458,94458,0,0,0,0,0,0,0,0,0,0


Silver candidate prepared — NOT published
root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- stay_id: integer (nullable = true)
 |-- first_careunit: string (nullable = true)
 |-- last_careunit: string (nullable = true)
 |-- intime: timestamp (nullable = true)
 |-- outtime: timestamp (nullable = true)
 |-- los: double (nullable = true)



In [0]:

# ============================================================
# STEP 4: CROSS-TABLE REFERENTIAL VALIDATION
# Read-only — no tables modified
# ============================================================

from pyspark.sql import functions as F

# Reference tables already published by the team
patients_ref = (
    spark.table(f"{SILVER}.patients")
    .select(
        F.col("subject_id").alias("p_subject_id")
    )
    .distinct()
)

admissions_ref = (
    spark.table(f"{SILVER}.admissions")
    .select(
        F.col("hadm_id").alias("a_hadm_id"),
        F.col("subject_id").alias("a_subject_id")
    )
    .distinct()
)

# Check patient and hospital admission relationships
linked = (
    icu_silver.alias("i")
    .join(
        patients_ref.alias("p"),
        F.col("i.subject_id") == F.col("p.p_subject_id"),
        "left"
    )
    .join(
        admissions_ref.alias("a"),
        F.col("i.hadm_id") == F.col("a.a_hadm_id"),
        "left"
    )
)

integrity = linked.agg(
    F.count("*").alias("joined_rows"),

    F.count_if(
        F.col("i.subject_id").isNull()
    ).alias("null_subject_ids"),

    F.count_if(
        F.col("i.hadm_id").isNull()
    ).alias("null_hadm_ids"),

    F.count_if(
        F.col("i.subject_id").isNotNull() &
        F.col("p.p_subject_id").isNull()
    ).alias("missing_patients"),

    F.count_if(
        F.col("i.hadm_id").isNotNull() &
        F.col("a.a_hadm_id").isNull()
    ).alias("missing_admissions"),

    F.count_if(
        F.col("a.a_hadm_id").isNotNull() &
        (~F.col("i.subject_id").eqNullSafe(
            F.col("a.a_subject_id")
        ))
    ).alias("patient_admission_mismatches")
)

display(integrity)


joined_rows,null_subject_ids,null_hadm_ids,missing_patients,missing_admissions,patient_admission_mismatches
94458,0,0,0,0,0


In [0]:

# ============================================================
# STEP 5: VALIDATION-GATED SILVER PUBLICATION
# Target: <target_schema>.icustays (CREATE OR REPLACE)
# ============================================================

from pyspark.sql import functions as F

EXPECTED_ROWS = 94458  # profiled
bronze_rows = icu_bronze.count()

# 1. Retrieve previously defined validation results
quality = validation.first().asDict()
links = integrity.first().asDict()

# 2. Structural conditions (always enforced)
assert quality["total_rows"] == bronze_rows, "Row count differs from Bronze"
assert quality["unique_stays"] == quality["total_rows"], "stay_id is not unique"
assert quality["null_stays"] == 0

for name, value in quality.items():
    if name.startswith("cast_fail_"):
        assert value == 0, f"{name}: {value}"

# 3. Profiled expectations (gated)
if ENFORCE_PROFILED_COUNTS:
    assert quality["total_rows"] == EXPECTED_ROWS

    for name in [
        "reversed_times",
        "negative_los",
        "los_mismatch_over_1min"
    ]:
        assert quality[name] == 0, f"{name}: {quality[name]}"

    # Referential integrity against Silver patients / admissions
    assert links["joined_rows"] == EXPECTED_ROWS

    for name in [
        "null_subject_ids",
        "null_hadm_ids",
        "missing_patients",
        "missing_admissions",
        "patient_admission_mismatches"
    ]:
        assert links[name] == 0, f"{name}: {links[name]}"

# 5. Add processing timestamp
icu_final = icu_silver.withColumn(
    "_silver_processed_at",
    F.current_timestamp()
)

print("All validation gates passed.")
print("Publishing:", TARGET)

# 6. CREATE OR REPLACE (downstream Gold depends on this table)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
icu_final.writeTo(TARGET).using("delta").createOrReplace()

print("SUCCESS: Silver ICU stays table published.")


All validation gates passed.
Publishing: cdr_mimiciv.silver.icustays
SUCCESS: Silver ICU stays table published.


In [0]:

# ============================================================
# STEP 6: FINAL SILVER TABLE VERIFICATION
# Read-only — no changes to any tables
# ============================================================

from pyspark.sql import functions as F

saved = spark.table(TARGET)

# 1. Validate published schema
expected_types = {
    "subject_id": "int",
    "hadm_id": "int",
    "stay_id": "int",
    "first_careunit": "string",
    "last_careunit": "string",
    "intime": "timestamp",
    "outtime": "timestamp",
    "los": "double",
    "_silver_processed_at": "timestamp"
}

actual_types = {
    field.name: field.dataType.simpleString()
    for field in saved.schema.fields
}

assert list(actual_types.items()) == list(expected_types.items()), (
    f"Schema mismatch: {actual_types}"
)

# 2. Check saved records and processing metadata
result = saved.agg(
    F.count("*").alias("silver_rows"),
    F.countDistinct("stay_id").alias("unique_stays"),
    F.count_if(F.col("stay_id").isNull())
        .alias("null_stays"),
    F.count_if(F.col("_silver_processed_at").isNull())
        .alias("missing_processing_timestamps")
).first().asDict()

# 3. Compare all 8 clinical columns in both directions
# exceptAll preserves duplicate-row semantics
cols = icu_silver.columns

missing_from_saved = (
    icu_silver.select(*cols)
    .exceptAll(saved.select(*cols))
    .count()
)

unexpected_in_saved = (
    saved.select(*cols)
    .exceptAll(icu_silver.select(*cols))
    .count()
)

# 4. Final quality gate
assert result["silver_rows"] == bronze_rows
assert result["unique_stays"] == result["silver_rows"]
if ENFORCE_PROFILED_COUNTS:
    assert result["silver_rows"] == EXPECTED_ROWS
assert result["null_stays"] == 0
assert result["missing_processing_timestamps"] == 0
assert missing_from_saved == 0
assert unexpected_in_saved == 0

print("=== FINAL ICUSTAYS AUDIT ===")
print(result)
print("Missing from saved:", missing_from_saved)
print("Unexpected in saved:", unexpected_in_saved)
print("FINAL STATUS: PASSED")


=== FINAL ICUSTAYS AUDIT ===
{'silver_rows': 94458, 'unique_stays': 94458, 'null_stays': 0, 'missing_processing_timestamps': 0}
Missing from saved: 0
Unexpected in saved: 0
FINAL STATUS: PASSED


In [0]:
dbutils.notebook.exit("icustays:OK")